In [2]:
from google.colab import drive
drive.mount('/content/drive')



ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Remove conflicting builds first
!pip uninstall -y protobuf cryptography flwr

# Install versions Flower wants
!pip install "cryptography>=44.0.1,<45.0.0" "protobuf>=4.21.6,<5.0.0"

# Then reinstall Flower and PyTorch
!pip install flwr==1.22.0 torch torchvision

In [ ]:
# Uninstall any conflicting versions
!pip uninstall -y flwr ray

# Install Flower with simulation extras + PyTorch
!pip install -U "flwr[simulation]" torch torchvision --quiet


In [ ]:
import ray
print(ray.__version__)


ModuleNotFoundError: No module named 'ray'

In [ ]:
#Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader
import copy
import warnings
from google.colab import files  # for download

warnings.filterwarnings("ignore", category=DeprecationWarning)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
# Dataset path

DATA_DIR = "/content/drive/MyDrive/Mod_data"  # Update if needed

In [ ]:
#Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(32 * 32 * 32, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 32 * 32)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
# Data Loaders (clients + test)

def get_client_loaders(data_dir=DATA_DIR, num_clients=2, batch_size=8):
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    total_size = len(dataset)
    part_size = total_size // num_clients

    sizes = [part_size]*(num_clients-1)
    sizes.append(total_size - sum(sizes))
    subsets = random_split(dataset, sizes)

    client_loaders = [DataLoader(subset, batch_size=batch_size, shuffle=True) for subset in subsets]

    # Test set (20% of total)
    test_size = int(0.2 * total_size)
    train_size = total_size - test_size
    _, test_data = random_split(dataset, [train_size, test_size])
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return client_loaders, test_loader

In [ ]:
#  Training Utilities

def train_one_client(model, train_loader, epochs=1, lr=0.01):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    total, correct = 0, 0
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            _, pred = torch.max(output, 1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    acc = correct / total
    return model.state_dict(), acc

def test_model(model, test_loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)
            _, pred = torch.max(output, 1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return correct / total

def fed_avg(models_state_dicts):
    avg_state_dict = copy.deepcopy(models_state_dicts[0])
    for key in avg_state_dict.keys():
        for i in range(1, len(models_state_dicts)):
            avg_state_dict[key] += models_state_dicts[i][key]
        avg_state_dict[key] = torch.div(avg_state_dict[key], len(models_state_dicts))
    return avg_state_dict

In [ ]:
global_acc = [52.5, 52.5, 95]

plt.bar(rounds, global_acc, color='skyblue')
plt.xlabel("Round")
plt.ylabel("Global Model Accuracy (%)")
plt.title("Global Model Accuracy per Round")
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_true = [0,0,1,1,0,1,0,1]   # 0: Normal, 1: Lung Cancer
y_pred = [0,0,1,0,0,1,1,1]

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal','Lung Cancer'], yticklabels=['Normal','Lung Cancer'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Example data
clients = ["Client 1", "Client 2"]
rounds = ["Round 1", "Round 2", "Round 3"]
accuracy = np.array([
    [52, 54, 60],  # Client 1
    [46, 40, 57]   # Client 2
])

plt.figure(figsize=(8,4))
sns.heatmap(accuracy, annot=True, fmt=".0f", cmap="YlGnBu", xticklabels=rounds, yticklabels=clients)
plt.title("Client Training Accuracy Heatmap")
plt.xlabel("Federated Learning Rounds")
plt.ylabel("Clients")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

images = ["/content/drive/MyDrive/Mod_data/LUNG CANCER/person10_bacteria_43.jpeg", "/content/drive/MyDrive/Mod_data/NORMAL/IM-0636-0001.jpeg"]
preds = ["Lung Cancer", "Normal"]

fig, axes = plt.subplots(1, len(images), figsize=(10,5))
for i, img_path in enumerate(images):
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(preds[i])
    axes[i].axis('off')
plt.show()


In [ ]:
#  Federated Training

def federated_training(rounds=3, num_clients=2):
    client_loaders, test_loader = get_client_loaders(num_clients=num_clients)
    global_model = CNNModel().to(DEVICE)

    for r in range(rounds):
        print(f"\n=== 🔹 Round {r+1} ===")
        client_states = []
        client_train_accs = []

        # Train each client
        for c in range(num_clients):
            local_model = copy.deepcopy(global_model).to(DEVICE)
            state_dict, train_acc = train_one_client(local_model, client_loaders[c])
            print(f"Client {c+1} Training Accuracy: {train_acc*100:.2f}%")
            client_states.append(state_dict)
            client_train_accs.append(train_acc)

        # FedAvg aggregation
        global_state_dict = fed_avg(client_states)
        global_model.load_state_dict(global_state_dict)

        # Test global model
        global_acc = test_model(global_model, test_loader)
        print(f"🌐 Global Model Accuracy after Round {r+1}: {global_acc*100:.2f}%")

    # Save global model locally in Colab
    MODEL_PATH = "federated_lung_model_manual.pth"
    torch.save(global_model.state_dict(), MODEL_PATH)
    print(f"\n✅ Global Model saved locally as {MODEL_PATH}")

    # Download model directly
    files.download(MODEL_PATH)

# Run Federated Training

if __name__ == "__main__":
    federated_training(rounds=3, num_clients=2)